In [0]:
print(spark)
print(type(spark))

<class 'pyspark.sql.connect.session.SparkSession'>


In [0]:
%sql
-- Create our project Volume inside workspace.default
-- Every project file lives under this path from here forward
CREATE VOLUME IF NOT EXISTS workspace.default.iot_pipeline

In [0]:
# Verify the Volume path resolves and is empty (we haven't written anything yet)
files = dbutils.fs.ls("/Volumes/workspace/default/iot_pipeline/")
print(f" Volume accessible. Contents: {files}")

# Also lock in our path as a constant — every cell from here forward uses this
VOLUME_BASE = "/Volumes/workspace/default/iot_pipeline"
RAW_DIR     = f"{VOLUME_BASE}/raw"
print(f" Raw data will land at: {RAW_DIR}")

 Volume accessible. Contents: []
 Raw data will land at: /Volumes/workspace/default/iot_pipeline/raw


In [0]:
from pyspark.sql import functions as F # This is Spark's built-in functions library such as F.rand(), F.when(), F.lit() etc, these are the distibuted, and they run seperately inside each partition

from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, FloatType, LongType, IntegerType)

# -- Storgae paths
volume_base = "/Volumes/workspace/default/iot_pipeline"
raw_dir = f"{volume_base}/raw"

# Scale Parameters - 12 million rows, 4 partitions, 4 output files, nearly 275 MB each, and each file stays under 300 MB, are large enough to avoid the small file problem

TOTAL_ROWS = 12000000
NUM_PARTITIONS = 4

# IOT-domain values - used for to build random categorical columns
REGIONS = ["North","South","East","West"]
STATUSES = ["OK","WARN","CRITICAL","OFFLINE"]
EVENT_TYPES = ["TEMP_READING","BATTERY_LOW","THRESHOLD_BREACHED","HEARTBEAT"]

# Error Injection rates - 
NULL_DEVICE_ID_RATE  = 0.005   # 0.5%  → unregistered packet
INVALID_TS_RATE      = 0.010   # 1.0%  → malformed timestamp from firmware bug
NULL_TEMP_RATE       = 0.020   # 2.0%  → sensor read failure
EXTREME_TEMP_RATE    = 0.010   # 1.0%  → hardware fault reporting -999 or 9999
NULL_HUMIDITY_RATE   = 0.020   # 2.0%  → humidity sensor offline
NULL_BATTERY_RATE    = 0.010   # 1.0%  → battery monitor fault

print("Imports loaded")
print(f"output path : {raw_dir}")
print(f"Total rows  : {TOTAL_ROWS}")
print(f"output files : {NUM_PARTITIONS}")
print(f"Error rates: device_id = {NULL_DEVICE_ID_RATE * 100}%)| "
      f"timestamp={INVALID_TS_RATE*100}% | "
      f"temp={NULL_TEMP_RATE*100}%")

Imports loaded
output path : /Volumes/workspace/default/iot_pipeline/raw
Total rows  : 12000000
output files : 4
Error rates: device_id = 0.5%)| timestamp=1.0% | temp=2.0%


In [0]:
import time
BASE_TS = 1_704_067_200  #Unix timestamp 
t_start = time.time()

df_raw = (spark
          .range(TOTAL_ROWS)
          .repartition(NUM_PARTITIONS)
          .select(

          #Column 1: Device_id
           F.when(
            F.rand(seed=1) < NULL_DEVICE_ID_RATE,
            F.lit(None).cast(StringType())            
        ).otherwise(                                  
            F.concat(                              
                F.lit("SENSOR_"),
                F.lpad(
                    (F.col("id") % 1000 + 1)     
                    .cast(StringType()), 4, "0"       
                )
            )
        ).alias("device_id"),

        # Column 2: Timestamp
        F.when(
            F.rand(seed=2) < INVALID_TS_RATE,
            F.lit("INVALID_DATE")                     
        ).otherwise(                                  
            F.from_unixtime(                          
                F.lit(BASE_TS) +
                (F.rand(seed=3) * 31_536_000)         
                .cast(LongType()),                   
                "yyyy-MM-dd HH:mm:ss"
            )
        ).alias("timestamp"),

        # Column 3: temperature_c
        F.when(
            F.rand(seed=4) < NULL_TEMP_RATE,
            F.lit(None).cast(FloatType())             
        ).when(
            F.rand(seed=5) < EXTREME_TEMP_RATE,
            F.when(F.rand(seed=6) < 0.5,
                   F.lit(-999.0).cast(FloatType()))   
             .otherwise(F.lit(9999.0).cast(FloatType()))  
        ).otherwise(
            (F.rand(seed=7) * 65 - 10)               
            .cast(FloatType())                    
        ).alias("temperature_c"),

        # Column 4 - Humdity_pct
         F.when(
            F.rand(seed=8) < NULL_HUMIDITY_RATE,
            F.lit(None).cast(FloatType())
        ).otherwise(
            (F.rand(seed=9) * 100).cast(FloatType())
        ).alias("humidity_pct"),

        # Column 5 - Region
        F.element_at(
            F.array([F.lit(r) for r in REGIONS]), 
            (F.floor(F.rand(seed=10) * len(REGIONS)) 
             + 1).cast(IntegerType())             
        ).alias("region"),

        # Column 6 - Status
        F.element_at(
            F.array([F.lit(s) for s in STATUSES]),
            (F.floor(F.rand(seed=11) * len(STATUSES))
             + 1).cast(IntegerType())
        ).alias("status"),

        # Column 7 - Battery level 
        F.when(
            F.rand(seed=12) < NULL_BATTERY_RATE,
            F.lit(None).cast(FloatType())
        ).otherwise(
            (F.rand(seed=13) * 100).cast(FloatType())
        ).alias("battery_level"),

        # Column 8 - Event type
        F.element_at(
            F.array([F.lit(e) for e in EVENT_TYPES]),
            (F.floor(F.rand(seed=14) * len(EVENT_TYPES))
             + 1).cast(IntegerType())
        ).alias("event_type")
          )
)

t_end = time.time()

df_raw.printSchema() # It shows column names and types but not the data
print(f" DataFrame defined in :{t_end-t_start:.3f} seconds")
# Write the DataFrame to the raw directory
print(f" Columns: {df_raw.columns}")
print(f" Target rows: {TOTAL_ROWS}")







root
 |-- device_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- temperature_c: float (nullable = true)
 |-- humidity_pct: float (nullable = true)
 |-- region: string (nullable = false)
 |-- status: string (nullable = false)
 |-- battery_level: float (nullable = true)
 |-- event_type: string (nullable = false)

 DataFrame defined in :0.046 seconds
 Columns: ['device_id', 'timestamp', 'temperature_c', 'humidity_pct', 'region', 'status', 'battery_level', 'event_type']
 Target rows: 12000000


In [0]:
import time
t_write_start = time.time()
(df_raw
 .write
 .mode("Overwrite")
 .option("header", "true")
 .csv(raw_dir)
 )
t_write_end = time.time()
write_duration = t_write_end - t_write_start
print(f" DataFrame written in :{write_duration:.1f} seconds")
print(f" Total duration :{(t_end-t_start+write_duration):.1f} seconds")
print(f" Cell 3 (generate + write) :{write_duration:.1f} seconds")


 
 
 
 
 

 DataFrame written in :19.1 seconds
 Total duration :19.2 seconds
 Cell 3 (generate + write) :19.1 seconds


In [0]:
# This cell is responsible for to verify the files on the disk

files = dbutils.fs.ls(raw_dir)
print(f"Files written to: {raw_dir}")
total_bytes = 0
csv_files = 0
for f in files:
    size_mb = f.size/(1024*1024)
    total_bytes += f.size
    if f.name.endswith(".csv"):
        csv_files +=1
        print(f"{f.name:<50} {size_mb:8.1f} MB")
    else:
        print(f"{f.name:<50} {'(marker)':>10}")

print(f"{'─' * 60}")
print(f" CSV files   : {csv_files}")
print(f" Total size  : {total_bytes / (1024**3):.2f} GB")



Files written to: /Volumes/workspace/default/iot_pipeline/raw
_SUCCESS                                             (marker)
_committed_3831666858391064843                       (marker)
_started_3831666858391064843                         (marker)
part-00000-tid-3831666858391064843-f24d5912-15f4-4f4a-8291-b36c4cd27dd0-137-1-c000.csv    244.4 MB
part-00001-tid-3831666858391064843-f24d5912-15f4-4f4a-8291-b36c4cd27dd0-138-1-c000.csv    244.5 MB
part-00002-tid-3831666858391064843-f24d5912-15f4-4f4a-8291-b36c4cd27dd0-139-1-c000.csv    244.4 MB
part-00003-tid-3831666858391064843-f24d5912-15f4-4f4a-8291-b36c4cd27dd0-140-1-c000.csv    244.4 MB
────────────────────────────────────────────────────────────
 CSV files   : 4
 Total size  : 0.95 GB
